# QASPER dev: BM25 vs real BGE Dense

Select T4 GPU, then run the four code cells in order. This isolated package
contains 18,587 original paragraph chunks and 393 development qrels. It excludes
the 417 test-candidate qrels, verifier datasets, model adapters and credentials.
It builds a NEW FAISS IndexFlatIP with normalized BGE-Large embeddings at the
pinned revision and verifies corpus/ID mapping/artifact hashes before scoring.
BM25 and Dense use the same dev questions, corpus, k=5 and no retrieval history.
Every query records the requested/actual backend; any fallback invalidates the
entire Dense score. A missing/invalid FAISS stops before loading the encoder.

This is QASPER-derived global paragraph retrieval with paper-title-prefixed
questions, not the official QASPER answer metric. Recall does not measure false
support or answer quality. Download the ZIP even if a stage fails. It includes
the index (~76 MB uncompressed), logs, dependencies, input hashes and traces.
Stop the GPU after download. Do not rerun stages in the same run directory.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import base64, hashlib, io, json, os, signal, subprocess, sys, time, uuid, zipfile
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
run_root = Path('/content') / ('qasper-dev-' + run_id)
root, out = run_root / 'workspace', run_root / 'output'
root.mkdir(parents=True, exist_ok=False)
out.mkdir(exist_ok=False)
payload = '...'
payload_sha256 = 'daa725644223196d5d47b907f690a8f03bedc013a91ffb9263133e46c123d1ea'
manifest = {'kb_indexing/data/qasper_eval_20260926/corpus.jsonl': {'size': 14185126, 'sha256': '4ad78d602384e8c50a02e0ab5c2363b26ecf74b06296a3457a41ef05ee6631c6'}, 'kb_indexing/data/qasper_eval_20260926/manifest.json': {'size': 4761, 'sha256': 'd47d47c9a6e83cac349645d69ac01bfe775fe52eaac045867b8093cec97b2a71'}, 'kb_indexing/data/qasper_eval_20260926/qrels_dev.jsonl': {'size': 222172, 'sha256': 'b61631d9cf2a3f0729de1168d13896fe056ba87dea0d7157c0c541a383d3225e'}, 'research_rag/__init__.py': {'size': 134, 'sha256': 'cc7fea0e7e9fe57abaea1ca3e78e70ecaad9feb9819b088c50cdf26ba0d249e5'}, 'research_rag/controller.py': {'size': 7426, 'sha256': 'b97466aece81f27bd89d62f1fd0c61857a8a604b2b9681faf1850cdfb55ff43f'}, 'research_rag/evaluation/__init__.py': {'size': 66, 'sha256': 'edf15e92c3397d31441733113a0ea6bc45d805a6084ef354526ffce327d7dd5d'}, 'research_rag/evaluation/qasper_benchmark.py': {'size': 11034, 'sha256': '283fc0c2b50f76bb72d95bab4076ba305542ae4cb6c97f1326ee8f6307136e6e'}, 'research_rag/graph.py': {'size': 32194, 'sha256': 'b2883b3b7c2ca75cda20ab9223fb8012591dcd3284dc2eca9595af083fc275df'}, 'research_rag/indexing.py': {'size': 5245, 'sha256': '2e02675e5094a278b4eb9d0e2eb0245389fda47ad0d9236ba69fef68f457cd14'}, 'research_rag/planning.py': {'size': 9967, 'sha256': '680b25c3d049f7fa908ebaaed9b2a799a2430425eefd33ffb2b362e36a2d24f7'}, 'research_rag/retrieval.py': {'size': 12796, 'sha256': 'f8fc0c3118c0feb7dcb845f6be2a3dc026f253679524cbdcb5c4ec9bc1f6c75b'}, 'research_rag/state.py': {'size': 1901, 'sha256': '0da55a10e4f5d1c33b4f06cad2e78553bfb190bc30c8b96b76a699e69a6fdc39'}, 'research_rag/tests/test_qasper_benchmark.py': {'size': 4323, 'sha256': 'f1e615cc0227deecbd39c9778b20c48d839099d9c6fc74ffbfc9adeac7c50462'}}
data = base64.b64decode(payload, validate=True)
assert hashlib.sha256(data).hexdigest() == payload_sha256
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    assert archive.testzip() is None and set(archive.namelist()) == set(manifest)
    for name in archive.namelist():
        relative = PurePosixPath(name)
        assert not relative.is_absolute() and '..' not in relative.parts and '\\' not in name
        raw = archive.read(name)
        assert len(raw) == manifest[name]['size'] and hashlib.sha256(raw).hexdigest() == manifest[name]['sha256']
        target = root / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)
(out / 'input-manifest.json').write_text(json.dumps(manifest, indent=2))
env = os.environ.copy()
env.update(PYTHONPATH=str(root), HF_HOME=str(run_root / 'hf'), OMP_NUM_THREADS='4', TOKENIZERS_PARALLELISM='false')
env.pop('HF_HUB_OFFLINE', None)
env.pop('TRANSFORMERS_OFFLINE', None)
venv = run_root / 'venv'
python = str(venv / 'bin/python')
statuses = {}
def bundle():
    # This dedicated directory contains only diagnostics produced by these cells.
    entries = [{'path':str(p.relative_to(out)), 'size':p.stat().st_size,
                'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
               for p in sorted(out.rglob('*')) if p.is_file() and p.name != 'output-manifest.json']
    (out/'output-manifest.json').write_text(json.dumps({'run_id':run_id,'files':entries},indent=2))
    archive = out.with_suffix('.zip')
    pending = archive.with_suffix('.zip.partial')
    with zipfile.ZipFile(pending, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file(): z.write(p, str(p.relative_to(out)))
    pending.replace(archive)
    print('Diagnostic archive:', archive.name, flush=True)

def run(name, command, *, timeout=1800, child_env=None, required=True):
    assert name not in statuses, 'Preserve the previous stage; start a fresh run for retries'
    started = time.perf_counter()
    status = {'state':'running'}
    statuses[name] = status
    (out/'status.json').write_text(json.dumps(statuses,indent=2))
    try:
        with (out/(name+'.stdout.log')).open('x') as stdout, (out/(name+'.stderr.log')).open('x') as stderr:
            process = subprocess.Popen(command, cwd=root, env=child_env or env,
                                       stdout=stdout, stderr=stderr, start_new_session=True)
            try:
                while True:
                    try:
                        code = process.wait(timeout=15)
                        break
                    except subprocess.TimeoutExpired:
                        elapsed = time.perf_counter()-started
                        print(name, 'running', round(elapsed), 'seconds', flush=True)
                        if elapsed > timeout: raise TimeoutError(name)
            except BaseException:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
                raise
        status.update(state='completed' if code == 0 else 'failed', exit_code=code)
    except BaseException as exc:
        status.update(state='interrupted', error_type=type(exc).__name__)
        raise
    finally:
        status['elapsed_ms'] = round((time.perf_counter()-started)*1000,2)
        (out/'status.json').write_text(json.dumps(statuses,indent=2))
        bundle()
    print(name, status, flush=True)
    if required and code:
        print((out / (name + '.stderr.log')).read_text()[-12000:], flush=True)
        raise RuntimeError(name + ' failed; inspect its saved stderr')
    return code
run('gpu-host', ['nvidia-smi'])
print('Run:', run_root)


In [ ]:
run('venv', [sys.executable, '-m', 'venv', '--without-pip', str(venv)])
run('bootstrap', [sys.executable, '-m', 'pip', '--python', python, 'install', '--no-cache-dir', 'pip==26.2.1'])
run('torch', [python, '-m', 'pip', 'install', '--no-cache-dir', 'torch==2.10.0', 'torchvision==0.25.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run('runtime', [python, '-m', 'pip', 'install', '--no-cache-dir', 'rank-bm25==0.2.2', 'faiss-cpu==1.15.0', 'sentence-transformers==6.0.0', 'transformers==5.16.1', 'pydantic==2.13.5'])
run('pip-check', [python, '-m', 'pip', 'check'])
run('pip-freeze', [python, '-m', 'pip', 'freeze'])
run('regression', [python, '-m', 'unittest', 'research_rag.tests.test_qasper_benchmark', '-v'])
run('cuda-check', [python, '-c', "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"])


In [ ]:
run('download-pinned-bge', [python, '-c', "from huggingface_hub import snapshot_download; from research_rag.retrieval import EMBEDDING_MODEL,EMBEDDING_REVISION; print(snapshot_download(EMBEDDING_MODEL, revision=EMBEDDING_REVISION, ignore_patterns=['*.bin','onnx/*','openvino/*']))"], timeout=1800)
data_dir = root / 'kb_indexing/data/qasper_eval_20260926'
index_dir = out / 'qasper-index'
run('build-qasper-faiss', [python, '-u', '-m', 'research_rag.indexing', '--corpus', str(data_dir / 'corpus.jsonl'), '--output-dir', str(index_dir), '--dense', '--device', 'cuda:0', '--batch-size', '8'], timeout=5400)


In [ ]:
assert all(statuses.get(name, {}).get('exit_code') == 0 for name in ('regression','cuda-check','build-qasper-faiss'))
code = run('paired-dev-benchmark', [python, '-u', '-m', 'research_rag.evaluation.qasper_benchmark', '--data-dir', str(data_dir), '--index-dir', str(index_dir), '--output-dir', str(out / 'benchmark'), '--mode', 'paired', '--top-k', '5'], timeout=3600, required=False)
summary = out / 'benchmark/summary.json'
print(summary.read_text() if summary.exists() else 'Incomplete; inspect saved logs.')
bundle()
print('A valid paired result requires 393 BM25 + 393 Dense queries, no fallback, matching hashes. No test candidate or verifier/E2E evaluation here.')
from google.colab import files
files.download(str(out.with_suffix('.zip')))
assert code == 0, 'Do not report Dense metrics from a failed or fallback run.'
